# COSC 4368 – Task 4: CNN-Based Clothing Classification
**Spring 2026 | Individual Assignment | PyTorch Version**

> **Setup (run once in your terminal):**
> ```
> pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
> pip install scikit-learn matplotlib seaborn pillow pandas jupyter
> ```
> Then launch with: `jupyter notebook`
>
> Place the dataset so that `clothing-dataset/images/` and `clothing-dataset/images.csv`
> exist in the same folder as this notebook — or update `DATA_DIR` in the Config cell.

**Code generated with assistance from Claude (Anthropic). Written analysis sections are the student's own work.**

In [ ]:
import os, time, warnings, json, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

warnings.filterwarnings('ignore')

# ── CONFIG ─────────────────────────────────────────────────────────────────────
DATA_DIR   = 'clothing-dataset'
IMG_DIR    = os.path.join(DATA_DIR, 'images')
CSV_PATH   = os.path.join(DATA_DIR, 'images.csv')

IMG_SIZE   = 128
BATCH_SIZE = 32
MAX_EPOCHS = 20
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# Use GPU if available (your RTX 3080 will show up here)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Section 1: Data Preprocessing and Basic CNN

### 1.1 Data Loading and Preprocessing

In [ ]:
# ── Load CSV ───────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Total records: {len(df)}")
print("\nClass distribution:")
print(df['label'].value_counts().to_string())

CLASSES     = sorted(df['label'].unique().tolist())
CLS2IDX     = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
print(f"\nClasses ({NUM_CLASSES}):", CLASSES)

In [ ]:
# ── Load all images into memory ────────────────────────────────────────────────
def load_images(df, img_dir, img_size):
    X, y = [], []
    skipped = 0
    for _, row in df.iterrows():
        path = os.path.join(img_dir, row['image'] + '.jpg')
        if not os.path.exists(path):
            skipped += 1; continue
        try:
            img = Image.open(path).convert('RGB').resize((img_size, img_size))
            X.append(np.array(img, dtype=np.float32) / 255.0)  # normalize to [0,1]
            y.append(CLS2IDX[row['label']])
        except:
            skipped += 1
    print(f"Loaded {len(X)} images | Skipped: {skipped}")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

X, y = load_images(df, IMG_DIR, IMG_SIZE)
print(f"X shape: {X.shape}  |  Pixel range: [{X.min():.2f}, {X.max():.2f}]")

In [ ]:
# ── Train / Validation / Test split: 70 / 15 / 15 ───────────────────────────────
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

print(f"Train : {len(X_train):>4}  ({len(X_train)/len(X)*100:.0f}%)")
print(f"Val   : {len(X_val):>4}  ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test  : {len(X_test):>4}  ({len(X_test)/len(X)*100:.0f}%)")

In [ ]:
# ── PyTorch Dataset class ─────────────────────────────────────────────────────
class ClothingDataset(Dataset):
    def __init__(self, X, y, transform=None):
        # PyTorch expects (N, C, H, W) — transpose from (N, H, W, C)
        self.X = torch.from_numpy(X.transpose(0, 3, 1, 2))
        self.y = torch.from_numpy(y)
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = self.X[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.y[idx]

# Augmentation transform (for Section 3.1 — defined here for reuse)
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(45),          # 45-degree rotation as required
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.1),
])

# No-augmentation datasets
train_ds = ClothingDataset(X_train, y_train)
val_ds   = ClothingDataset(X_val,   y_val)
test_ds  = ClothingDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

In [ ]:
# ── Visualize one sample per category ─────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Sample Images – One Per Category', fontsize=16, fontweight='bold')
for idx, cls in enumerate(CLASSES):
    sample_idx = np.where(y == idx)[0][0]
    ax = axes[idx // 5][idx % 5]
    ax.imshow(X[sample_idx])
    ax.set_title(cls, fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.savefig('fig1_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

### 1.2 Basic CNN Architecture

In [ ]:
class BasicCNN(nn.Module):
    """
    Basic CNN Architecture:
    - 2 Convolutional blocks with ReLU activation
    - Max Pooling after each conv block
    - Batch Normalization for training stability
    - Dropout (0.25 in conv blocks, 0.5 in dense) for regularization
    - Global Average Pooling to reduce parameters
    - Dense output with 10 units (Softmax applied in loss function)
    """
    def __init__(self, num_classes=10):
        super(BasicCNN, self).__init__()

        # ── Block 1 ──────────────────────────────────────────────────────────
        self.block1 = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        # ── Block 2 ──────────────────────────────────────────────────────────
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )

        # ── Global Average Pooling + Dense head ──────────────────────────────
        self.gap = nn.AdaptiveAvgPool2d(1)      # output: (batch, 64, 1, 1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x                                # raw logits (CrossEntropyLoss handles softmax)


# Instantiate and inspect
basic_model = BasicCNN(num_classes=NUM_CLASSES).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in basic_model.parameters())
trainable    = sum(p.numel() for p in basic_model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print()
print(basic_model)

### 1.3 Model Training

In [ ]:
# ── Training and evaluation helper functions ───────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs     = model(imgs)
            loss        = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total


def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


def train_model(model, train_loader, val_loader, name,
                lr=1e-3, max_epochs=MAX_EPOCHS, patience=5):
    """Full training loop with early stopping and LR scheduling."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val_acc   = 0.0
    best_weights   = None
    epochs_no_improve = 0

    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = evaluate_loader(model, val_loader, criterion)
        scheduler.step(vl_loss)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        print(f"Epoch {epoch:>2}/{max_epochs}  "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
              f"val_loss={vl_loss:.4f}  val_acc={vl_acc:.4f}")

        # Save best model
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, f'best_{name}.pth')
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (patience={patience})")
                break

    elapsed = time.time() - t0
    model.load_state_dict(best_weights)      # restore best weights
    print(f"\nDone. Best val_acc={best_val_acc:.4f}  Time={elapsed:.1f}s  Epochs={epoch}")
    return history, elapsed

In [ ]:
# ── Train the basic CNN ────────────────────────────────────────────────────────
print("Training Basic CNN...")
hist_basic, time_basic = train_model(
    basic_model, train_loader, val_loader, 'BasicCNN')

In [ ]:
# ── Plot training curves ───────────────────────────────────────────────────────
def plot_training_curves(history, title, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{title} – Training Curves', fontsize=14, fontweight='bold')

    ax1.plot(history['train_acc'], label='Train', color='steelblue', linewidth=2)
    ax1.plot(history['val_acc'],   label='Val',   color='orange',    linewidth=2)
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_loss'], label='Train', color='steelblue', linewidth=2)
    ax2.plot(history['val_loss'],   label='Val',   color='orange',    linewidth=2)
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_training_curves(hist_basic, 'Basic CNN', 'fig2_basic_curves.png')

---
## Section 2: Evaluation and Analysis

### 2.1 Performance Metrics

In [ ]:
# ── Full evaluation function ───────────────────────────────────────────────────
def full_evaluate(model, loader, class_names, model_name='Model'):
    criterion  = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate_loader(model, loader, criterion)
    y_pred, y_true      = get_predictions(model, loader)
    f1_w = f1_score(y_true, y_pred, average='weighted')

    print(f"{'='*55}")
    print(f"  {model_name} – Test Set Evaluation")
    print(f"{'='*55}")
    print(f"  Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")
    print(f"  Test Loss     : {test_loss:.4f}")
    print(f"  Weighted F1   : {f1_w:.4f}")
    print(f"{'='*55}\n")
    print("Per-Class Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    return y_pred, y_true, test_acc, f1_w

y_pred_basic, y_true_basic, acc_basic, f1_basic = full_evaluate(
    basic_model, test_loader, CLASSES, 'Basic CNN')

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    # Most confused pairs
    cm_nd = cm.copy(); np.fill_diagonal(cm_nd, 0)
    top   = np.unravel_index(np.argsort(cm_nd.ravel())[-5:], cm.shape)
    print("Top 5 most confused pairs:")
    for ti, pi in zip(top[0][::-1], top[1][::-1]):
        if cm_nd[ti, pi] > 0:
            print(f"  True: {class_names[ti]:<12} → Pred: {class_names[pi]:<12} ({cm_nd[ti,pi]} times)")

plot_confusion_matrix(y_true_basic, y_pred_basic, CLASSES,
                      'Confusion Matrix – Basic CNN',
                      'fig3_confusion_matrix_basic.png')

In [ ]:
# ── 5 Correct predictions (one per class) ─────────────────────────────────────
correct_mask = (y_pred_basic == y_true_basic)
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Correct Predictions – One Per Class', fontsize=14, fontweight='bold')
for idx, cls in enumerate(CLASSES):
    ok = np.where(correct_mask & (y_true_basic == idx))[0]
    ax = axes[idx // 5][idx % 5]
    if len(ok):
        ax.imshow(X_test[ok[0]])
        ax.set_title(f'✓ {cls}', fontsize=9, color='green', fontweight='bold')
    else:
        ax.set_title(f'{cls}\n(none correct)', fontsize=9, color='gray')
    ax.axis('off')
plt.tight_layout()
plt.savefig('fig4_correct_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10 Misclassified examples ─────────────────────────────────────────────────
wrong_idx = np.where(~correct_mask)[0]
rng = np.random.default_rng(SEED)
chosen = rng.choice(wrong_idx, size=min(10, len(wrong_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('10 Misclassified Examples', fontsize=14, fontweight='bold')
for pi, i in enumerate(chosen):
    ax = axes[pi // 5][pi % 5]
    ax.imshow(X_test[i])
    ax.set_title(f'True: {CLASSES[y_true_basic[i]]}\nPred: {CLASSES[y_pred_basic[i]]}',
                 fontsize=8, color='red')
    ax.axis('off')
plt.tight_layout()
plt.savefig('fig5_misclassified.png', dpi=150, bbox_inches='tight')
plt.show()

### Written Analysis – Misclassification Patterns

The confusion matrix for the Basic CNN reveals clear patterns in how the model struggles with this dataset. The most frequently confused pairs involve visually similar garment types: Shirt was the most over-predicted class, absorbing misclassifications from Longsleeve, T-Shirt, and Dress — all of which share a similar upper-body torso shape. This reflects the model's reliance on coarse silhouette features rather than finer details like sleeve length or collar style. Pants was the weakest class with zero correct predictions, likely because the model had not yet developed feature detectors sensitive enough to distinguish leg coverage at this early training stage. Shoes and Hat were among the better-performing categories because their shapes are visually distinctive and geometrically consistent across real-world images, producing strong and reliable activation patterns even in shallow convolutional layers. The "Not sure" category was unsurprisingly difficult — it contains inherently ambiguous images with no consistent visual pattern, making it essentially unlearnable as a coherent class. Overall, the 19.33% test accuracy reflects the challenge of learning from only 70 images per class with no augmentation and a relatively shallow architecture.

---
## Section 3: Advanced Techniques

### 3.1 Data Augmentation

In [ ]:
# ── Visualize augmentation examples ───────────────────────────────────────────
sample_img = X_train[0]    # shape (H, W, 3), float32 numpy

# Convert to PIL for torchvision transforms
to_pil    = transforms.ToPILImage()
to_tensor = transforms.ToTensor()
aug_vis   = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(45),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.1),
])

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Data Augmentation Examples\n(45° rotation + flip + zoom + brightness)',
             fontsize=13, fontweight='bold')

# Original
axes[0][0].imshow(sample_img)
axes[0][0].set_title('Original', fontsize=10, fontweight='bold')
axes[0][0].axis('off')

# 9 augmented versions
pil_img = to_pil((sample_img * 255).astype(np.uint8))
for i in range(1, 10):
    aug_pil = aug_vis(pil_img)
    r, c = i // 5, i % 5
    axes[r][c].imshow(aug_pil)
    axes[r][c].set_title(f'Augmented #{i}', fontsize=9)
    axes[r][c].axis('off')

plt.tight_layout()
plt.savefig('fig6_augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Augmented Dataset and DataLoader ──────────────────────────────────────────
# The aug_transform was defined earlier; we apply it via a new Dataset
train_ds_aug = ClothingDataset(X_train, y_train, transform=aug_transform)
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0, pin_memory=True)

# ── Augmented CNN — same architecture, trained with augmented data ─────────────
class AugmentedCNN(nn.Module):
    """Identical architecture to BasicCNN — augmentation is applied by the DataLoader."""
    def __init__(self, num_classes=10):
        super(AugmentedCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.gap(self.block2(self.block1(x))))

aug_model = AugmentedCNN(num_classes=NUM_CLASSES).to(DEVICE)
print("Training Augmented CNN (same architecture, augmented data)...")
hist_aug, time_aug = train_model(aug_model, train_loader_aug, val_loader, 'AugmentedCNN')

In [ ]:
# ── Evaluate and compare ──────────────────────────────────────────────────────
y_pred_aug, y_true_aug, acc_aug, f1_aug = full_evaluate(
    aug_model, test_loader, CLASSES, 'Augmented CNN')

# Side-by-side comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Basic CNN vs Augmented CNN – Validation Comparison',
             fontsize=14, fontweight='bold')
ax1.plot(hist_basic['val_acc'], label='Basic CNN',     color='steelblue', linewidth=2)
ax1.plot(hist_aug['val_acc'],   label='Augmented CNN', color='green',     linewidth=2)
ax1.set_title('Validation Accuracy'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(hist_basic['val_loss'], label='Basic CNN',     color='steelblue', linewidth=2)
ax2.plot(hist_aug['val_loss'],   label='Augmented CNN', color='green',     linewidth=2)
ax2.set_title('Validation Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig7_basic_vs_aug.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Basic CNN    : acc={acc_basic:.4f}  f1={f1_basic:.4f}")
print(f"Augmented CNN: acc={acc_aug:.4f}  f1={f1_aug:.4f}")
print(f"Difference   : {acc_aug - acc_basic:+.4f}")

### Written Analysis – Effect of Data Augmentation

Contrary to the typical expectation, data augmentation did not improve performance in this run — the Augmented CNN achieved a lower test accuracy (16.67%) and weighted F1 (0.1009) than the Basic CNN (19.33%, F1 0.1633). Several factors explain this outcome. First, with only 70 training images per class, the augmentation pipeline — which applied aggressive 45° rotations, random resized crops, and horizontal flips — may have distorted the limited training signal rather than enriching it, effectively making each batch harder to learn from before the model had established stable feature representations. Second, early stopping triggered after only 7 epochs for the Augmented CNN versus 9 for the Basic CNN, suggesting the augmented model's validation loss was noisier and converged to a worse local minimum. Third, several classes (Dress, Longsleeve, Not sure, Outwear, Pants, T-Shirt) had zero recall under augmentation, indicating the model collapsed to predicting only a few dominant classes (Shirt, Shoes, Shorts) under the heavier augmentation regime. This highlights an important practical lesson: augmentation is most effective when paired with sufficient base data and a model complex enough to generalize from the augmented signal. At this dataset scale, lighter augmentation or a longer training schedule would likely be needed to see the expected benefit.

### 3.2 Architecture Comparison – Four Experiments

In [ ]:
# ── Configurable deep CNN builder ─────────────────────────────────────────────
class DeepCNN(nn.Module):
    """
    Configurable 4-layer CNN.
    filters      : list of 4 filter counts
    kernel_sizes : list of 4 kernel sizes (3 or 5)
    dense_units  : units in fully-connected head
    """
    def __init__(self, filters, kernel_sizes, dense_units, num_classes=10):
        super(DeepCNN, self).__init__()
        assert len(filters) == 4 and len(kernel_sizes) == 4

        def conv_block(in_ch, out_ch, k):
            pad = k // 2          # same padding
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, k, padding=pad),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        # 4 conv layers, MaxPool after every 2nd
        self.conv1  = conv_block(3,          filters[0], kernel_sizes[0])
        self.conv2  = conv_block(filters[0], filters[1], kernel_sizes[1])
        self.pool1  = nn.Sequential(nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))

        self.conv3  = conv_block(filters[1], filters[2], kernel_sizes[2])
        self.conv4  = conv_block(filters[2], filters[3], kernel_sizes[3])
        self.pool2  = nn.Sequential(nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters[3], dense_units),
            nn.BatchNorm1d(dense_units),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(dense_units, num_classes),
        )

    def forward(self, x):
        x = self.pool1(self.conv2(self.conv1(x)))
        x = self.pool2(self.conv4(self.conv3(x)))
        return self.classifier(self.gap(x))


# ── 4 experiment configs ───────────────────────────────────────────────────────
EXPERIMENTS = [
    ('Deep_3x3',  [32, 64,128,256], [3,3,3,3], 256, 'Deep, all 3×3 kernels (VGG-style)'),
    ('Deep_5x5',  [32, 64,128,256], [5,5,5,5], 256, 'Deep, all 5×5 kernels (larger receptive field)'),
    ('Wide_3x3',  [64,128,256,512], [3,3,3,3], 512, 'Wide, 3×3 kernels, double filter counts'),
    ('Mixed_ker', [32, 64,128,256], [3,5,3,5], 256, 'Alternating 3×3 and 5×5 kernels'),
]

print("Architecture parameter counts:")
for name, filt, kern, dense, desc in EXPERIMENTS:
    m = DeepCNN(filt, kern, dense, NUM_CLASSES)
    params = sum(p.numel() for p in m.parameters())
    print(f"  {name:<12}: {params:>10,}  — {desc}")

In [ ]:
# ── Train all 4 experiments ────────────────────────────────────────────────────
# With an RTX 3080 each model trains in roughly 1-3 minutes.

all_results = [
    {'name':'BasicCNN',     'params': sum(p.numel() for p in basic_model.parameters()),
     'time': time_basic, 'acc': acc_basic, 'f1': f1_basic,
     'epochs': len(hist_basic['train_acc'])},
    {'name':'AugmentedCNN', 'params': sum(p.numel() for p in aug_model.parameters()),
     'time': time_aug,   'acc': acc_aug,   'f1': f1_aug,
     'epochs': len(hist_aug['train_acc'])},
]
all_hists  = [('BasicCNN', hist_basic), ('AugmentedCNN', hist_aug)]

for exp_name, filt, kern, dense, desc in EXPERIMENTS:
    print(f"\n{'─'*60}")
    print(f"Experiment: {exp_name}  —  {desc}")
    print(f"{'─'*60}")

    m = DeepCNN(filt, kern, dense, NUM_CLASSES).to(DEVICE)
    h, elapsed = train_model(m, train_loader, val_loader, exp_name)

    # Also plot individual curves
    plot_training_curves(h, exp_name, f'fig_curves_{exp_name}.png')

    yp, yt, acc, f1 = full_evaluate(m, test_loader, CLASSES, exp_name)
    params = sum(p.numel() for p in m.parameters())

    all_results.append({'name': exp_name, 'params': params,
                        'time': elapsed,  'acc': acc, 'f1': f1,
                        'epochs': len(h['train_acc'])})
    all_hists.append((exp_name, h))

print("\n✓ All experiments complete!")

In [ ]:
# ── Results comparison table ──────────────────────────────────────────────────
print("\n" + "="*80)
print(f"{'Model':<20} {'Parameters':>12} {'Time(s)':>10} {'Epochs':>8} {'Test Acc':>10} {'F1':>8}")
print("="*80)
for r in all_results:
    print(f"{r['name']:<20} {r['params']:>12,} {r['time']:>10.1f} "
          f"{r['epochs']:>8} {r['acc']:>10.4f} {r['f1']:>8.4f}")
print("="*80)

# Bar chart comparison
colors = ['steelblue','green','tomato','mediumpurple','darkorange','saddlebrown']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Architecture Comparison – All Models', fontsize=15, fontweight='bold')

names  = [r['name']   for r in all_results]
accs   = [r['acc']    for r in all_results]
f1s    = [r['f1']     for r in all_results]
params = [r['params'] for r in all_results]

axes[0].bar(names, [a*100 for a in accs], color=colors)
axes[0].set_title('Test Accuracy (%)'); axes[0].set_ylim(0, 100)
axes[0].set_xticklabels(names, rotation=30, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(accs): axes[0].text(i, v*100+0.5, f'{v*100:.1f}%', ha='center', fontsize=9)

axes[1].bar(names, f1s, color=colors)
axes[1].set_title('Weighted F1-Score'); axes[1].set_ylim(0, 1)
axes[1].set_xticklabels(names, rotation=30, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(f1s): axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=9)

axes[2].bar(names, [p/1000 for p in params], color=colors)
axes[2].set_title('Parameters (thousands)')
axes[2].set_xticklabels(names, rotation=30, ha='right')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fig8_architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── All validation curves on one plot ─────────────────────────────────────────
colors = ['steelblue','green','tomato','mediumpurple','darkorange','saddlebrown']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('All Models – Validation Metrics Over Training', fontsize=14, fontweight='bold')
for (name, h), c in zip(all_hists, colors):
    ax1.plot(h['val_acc'],  label=name, color=c, linewidth=2)
    ax2.plot(h['val_loss'], label=name, color=c, linewidth=2)
ax1.set_title('Validation Accuracy'); ax1.set_xlabel('Epoch')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax2.set_title('Validation Loss'); ax2.set_xlabel('Epoch')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig9_all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4: Final Analysis

### 4.1 Summary – All Models Compared

| Model | Parameters | Time (s) | Epochs | Test Acc | F1 |
|---|---|---|---|---|---|
| BasicCNN | 85,482 | 2.5 | 9 | 19.33% | 0.1633 |
| AugmentedCNN | 85,482 | 6.2 | 7 | 16.67% | 0.1009 |
| Deep_3x3 | 458,250 | 8.7 | 14 | 23.33% | 0.2225 |
| Deep_5x5 | 1,147,914 | 18.9 | 18 | **24.00%** | 0.2060 |
| Wide_3x3 | 1,821,706 | 16.6 | 10 | 23.33% | 0.2208 |
| Mixed_ker | 1,015,306 | 14.0 | 15 | 18.67% | 0.1587 |

The results reveal a consistent trend: deeper architectures with more convolutional layers outperformed the shallow BasicCNN. The best test accuracy came from Deep_5x5 (24.00%), closely followed by Deep_3x3 and Wide_3x3 (both 23.33%). The additional convolutional layers allowed these models to learn more hierarchical and discriminative feature representations, which is critical when classifying real-world clothing images with varied backgrounds and lighting. The AugmentedCNN underperformed the BasicCNN — counterintuitively — because aggressive augmentation on a very small dataset (70 images/class) degraded rather than enriched the training signal, causing several classes to drop to zero recall. The Mixed_ker model was the weakest of the deep architectures, suggesting that alternating kernel sizes without a principled strategy did not help compared to a uniform approach. Notably, all models trained in under 20 seconds, demonstrating the speed advantage of GPU training even at this scale.

### 4.2 Easier vs. Harder Categories

**Easiest categories:** Shoes was consistently the best-classified class across most models, with recall reaching 0.667 in several experiments. Its distinctive shape — a sole, toe box, and heel — produces strong and consistent low-level feature activations that even shallow networks can detect. Hat was similarly strong, benefiting from a consistent circular brim structure. Shirt and Shorts also performed relatively well in most models, likely because Shirt images were large in frame and Shorts have a distinctive silhouette with exposed legs.

**Hardest categories:** Pants achieved zero correct predictions in the BasicCNN entirely, with near-zero performance across most models. This may reflect that full-length trouser images are harder to distinguish from Longsleeve tops when images are inconsistently framed or cropped. Not sure was the most reliably poor performer — by definition this label covers ambiguous items with no consistent visual pattern, making it unlearnable as a distinct class. Longsleeve and T-Shirt were frequently confused with Shirt since all three are upper-body garments that differ only in sleeve length and collar detail — features that require fine-grained, high-resolution feature maps to resolve reliably.

### 4.3 Challenges and Solutions

**Small dataset size:** The most significant challenge was the limited data — only 700 training images across 10 classes (70 per class). This is far below what CNNs typically require to converge to strong solutions. The primary mitigation was applying Dropout regularization (0.25 in convolutional blocks, 0.50 in the dense head) and using early stopping with patience=5 to prevent over-training on noise.

**Augmentation backfiring:** Data augmentation, which typically improves generalization, actually hurt performance in this run. With such a small dataset, aggressive 45° rotations and resized crops appeared to add too much noise before the model had established useful features. A more effective approach would be to apply lighter augmentation (e.g., horizontal flip only) or to use augmentation only after a warm-up phase.

**Class confusion among similar garments:** Shirt, T-Shirt, and Longsleeve proved nearly indistinguishable for shallow models. Deeper architectures (Deep_3x3, Deep_5x5) partially addressed this by learning more discriminative higher-level features across four convolutional layers, improving accuracy by roughly 4–5 percentage points over the BasicCNN.

**Training instability:** Validation accuracy fluctuated significantly epoch-to-epoch across all models, reflecting the high variance inherent in small-batch evaluation on 150 test samples. The ReduceLROnPlateau scheduler helped stabilize later training epochs by halving the learning rate when validation loss stalled.

### 4.4 Key Insights and Recommendations

1. **Depth matters more than augmentation at this scale.** The four-layer deep architectures (Deep_3x3, Deep_5x5, Wide_3x3) consistently outperformed both the BasicCNN and the AugmentedCNN. More layers enabled learning richer feature hierarchies from the limited data, whereas augmentation without sufficient base examples was counterproductive.

2. **Kernel size had limited impact.** Deep_5x5 edged out Deep_3x3 by only 0.67 percentage points despite having 2.5× more parameters and 2× the training time. For 128×128 clothing images, 3×3 kernels efficiently capture the local textures and edges needed for classification without the added computational cost of 5×5 filters.

3. **The "Not sure" class is a structural problem.** Any model trained on this dataset will struggle with the "Not sure" label because it is definitionally ambiguous — it aggregates images that human annotators could not confidently categorize. Removing this class or relabeling these images would likely improve overall accuracy noticeably.

4. **Transfer learning is the clear next step.** All six models trained from scratch reached accuracies between 16–24% on 700 training images. A pretrained backbone such as MobileNetV2 or EfficientNet-B0, fine-tuned on this dataset, would almost certainly push accuracy above 60–70% by leveraging ImageNet feature detectors that already recognize textures, edges, and shapes relevant to clothing classification.

---
## Loading a Saved Model

In [ ]:
# ── How to reload any saved model ─────────────────────────────────────────────
# Models are saved as .pth files (PyTorch state dictionaries).

# Example: reload BasicCNN
loaded_model = BasicCNN(num_classes=NUM_CLASSES).to(DEVICE)
loaded_model.load_state_dict(torch.load('best_BasicCNN.pth', map_location=DEVICE))
loaded_model.eval()
print("BasicCNN loaded successfully!")

# Verify predictions match
yp_verify, yt_verify = get_predictions(loaded_model, test_loader)
verify_acc = (yp_verify == yt_verify).mean()
print(f"Verification accuracy: {verify_acc:.4f}")

# To load other models:
# aug_reload  = AugmentedCNN(num_classes=NUM_CLASSES).to(DEVICE)
# aug_reload.load_state_dict(torch.load('best_AugmentedCNN.pth', map_location=DEVICE))
#
# deep_reload = DeepCNN([32,64,128,256],[3,3,3,3],256,NUM_CLASSES).to(DEVICE)
# deep_reload.load_state_dict(torch.load('best_Deep_3x3.pth', map_location=DEVICE))